# Fact-Checking Pipeline — Google Colab (self-contained)

```
raw text ─▶ 🟦 TASK 1  Claim Detection    RoBERTa fine-tuned on ClaimBuster      → which sentences are claims
         ─▶ 🟨 TASK 2  Evidence Retrieval BM25 → bi-encoder → cross-encoder      → top-5 Wikipedia sentences
         ─▶ 🟥 TASK 3  Verification       DeBERTa NLI                            → SUPPORTS / REFUTES / NOT ENOUGH INFO
```

All the code is in this notebook; nothing is cloned or imported from the project repo.

**How to run**
1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all**.
   - First run: downloads FEVER (`wiki-pages.zip`, 1.7 GB) and builds the Wikipedia subset (~30 min).
     If there is no Task 1 model in Drive yet, it is trained here too (~15 min).
   - Later runs: everything already in Drive is skipped, so it only loads the models (~2 min).

**Everything persistent is stored in Google Drive**, in `DRIVE_DIR` (set in the first SETUP cell,
default `MyDrive/NLP/Praktikum/Final Project/`):

| Drive folder | Contents |
|---|---|
| `models/task1_roberta-base_label/` | trained Task 1 model (+ `metrics.json`) |
| `hf_cache/` | pretrained Hugging Face models (DeBERTa NLI, MiniLM bi-encoder, MiniLM cross-encoder, roberta-base) |
| `data/raw/` | FEVER claims, `wiki-pages.zip`, ClaimBuster CSVs |
| `data/processed/` | Task 2 corpus subset (`wiki_subset.jsonl`, `claims_sample.jsonl`), Task 1 splits |
| `outputs/` | evaluation metrics + predictions (`task2/`, `task3/`) and the last pipeline run |

**Task 1 model.** If you already trained it, put it anywhere inside `models/` (any subfolder), or put the zip
`task1_roberta-base_label.zip` from the old Task 1 notebook in `DRIVE_DIR`. Section 1.3 searches `models/` for a
folder with `model.safetensors` + `config.json` and uses it; it only trains a new model when it finds none.

Cells marked **(optional)** are evaluations; switch them off with their `RUN_...` checkbox for a quick demo.

# ⚙️ SETUP

In [1]:
# ---------- Config: every persistent path is in Google Drive ----------
import os
from pathlib import Path

DRIVE_DIR     = Path('/content/drive/MyDrive/NLP/Praktikum/Final Project')   # change if your folder is elsewhere
DATA_DIR      = DRIVE_DIR / 'data'
RAW_DIR       = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR    = DRIVE_DIR / 'models'
OUTPUTS_DIR   = DRIVE_DIR / 'outputs'
HF_CACHE_DIR  = DRIVE_DIR / 'hf_cache'

# Hugging Face models are cached in Drive, so they are downloaded only once.
# Must be set before transformers / sentence-transformers are imported.
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [2]:
# ---------- Mount Google Drive ----------
from google.colab import drive
drive.mount('/content/drive')

for d in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, OUTPUTS_DIR, HF_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
!ls -la "{DRIVE_DIR}"

Mounted at /content/drive
total 96
drwx------ 7 root root  4096 Nov 21 14:38 .
drwx------ 4 root root  4096 Sep 27 06:20 ..
drwx------ 4 root root  4096 Nov 21 15:00 data
-rw------- 1 root root 69470 Nov 21 15:22 fact_checking_colab.ipynb
drwx------ 2 root root  4096 Nov 21 15:00 hf_cache
drwx------ 2 root root  4096 Nov 21 14:40 model
drwx------ 2 root root  4096 Nov 21 15:00 models
drwx------ 2 root root  4096 Nov 21 15:00 outputs


In [3]:
# ---------- GPU check ----------
!nvidia-smi -L
import torch
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU (it still runs on CPU, but slowly)')

GPU 0: Tesla T4 (UUID: GPU-10c1cb72-0969-7ca4-12fc-638eee1233ba)
cuda available: True


In [4]:
# ---------- Dependencies (Colab already has torch, transformers, scikit-learn, pandas, spaCy) ----------
!pip install -q "sentence-transformers>=3.0" accelerate sentencepiece protobuf
!python -m spacy download en_core_web_sm 2>&1 | tail -1

'Restart kernel' or 'Restart runtime' option.


In [5]:
# ---------- Shared imports + helpers ----------
import http.client
import json
import random
import re
import shutil
import time
import unicodedata
import urllib.error
import urllib.request
import zipfile
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
import sentence_transformers
from tqdm.auto import tqdm


def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def stage(title):
    """Prints a clear marker in the cell output for each pipeline stage."""
    line = '━' * 90
    print(f'\n{line}\n  {title}\n{line}')


DEVICE = get_device()
print('device:', DEVICE, '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| sentence-transformers', sentence_transformers.__version__)

device: cuda | torch 2.14.1+cu130 | transformers 5.22.1 | sentence-transformers 5.7.0


# 📦 DATA · Download helpers
Shared by all tasks. Each file is streamed to `<name>.part` and renamed only when the byte count matches,
so an interrupted download is never mistaken for a finished one. Files already in Drive are re-checked
(zip directory, last JSON line) and downloaded again only if they look truncated.

In [6]:
FEVER_MIRRORS = [
    'https://fever.ai/download/fever',
    'https://s3-eu-west-1.amazonaws.com/fever.public',   # fallback; returned HTTP 403 on 2026-11-16
]
CLAIMBUSTER_BASE = 'https://zenodo.org/records/3609356/files'
CLAIMBUSTER_FILES = ['groundtruth.csv', 'crowdsourced.csv']
CLAIMBUSTER_DIR = RAW_DIR / 'claimbuster'

CHUNK_SIZE = 1 << 20
USER_AGENT = 'fact-checking-pipeline/0.1'


class DownloadError(RuntimeError):
    pass


def looks_complete(path):
    """Cheap truncation check that does not need the expected size."""
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return False
    if path.suffix == '.zip':
        return zipfile.is_zipfile(path)      # a truncated zip has no end-of-central-directory record
    if path.suffix == '.jsonl':
        return _last_line_is_json(path)
    return True


def _last_line_is_json(path):
    with open(path, 'rb') as f:
        f.seek(0, os.SEEK_END)
        end = f.tell()
        f.seek(max(0, end - CHUNK_SIZE))
        tail = f.read().rstrip(b'\n')
    last = tail.rsplit(b'\n', 1)[-1]
    try:
        json.loads(last)
    except ValueError:
        return False
    return True


def _stream_to_file(url, dest):
    part = dest.with_name(dest.name + '.part')
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
    with urllib.request.urlopen(request, timeout=60) as response:
        expected = response.headers.get('Content-Length')
        expected = int(expected) if expected is not None else None
        written = 0
        with open(part, 'wb') as out, tqdm(total=expected, unit='B', unit_scale=True, desc=dest.name) as bar:
            while True:
                chunk = response.read(CHUNK_SIZE)
                if not chunk:
                    break
                out.write(chunk)
                written += len(chunk)
                bar.update(len(chunk))

    if written == 0:
        part.unlink(missing_ok=True)
        raise DownloadError('empty response body')
    if expected is not None and written != expected:
        raise DownloadError(f'got {written} bytes, expected {expected}. Partial file kept at {part}')
    part.replace(dest)
    if not looks_complete(dest):
        raise DownloadError(f'{dest} was downloaded but fails the integrity check')
    return dest


def download_file(urls, dest, force=False):
    """Try each URL in order. Raise DownloadError if all of them fail."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and not force:
        if looks_complete(dest):
            return dest
        print(f'{dest} looks truncated, downloading again')

    errors = []
    for url in urls:
        print(f'Download {url} -> {dest}')
        try:
            return _stream_to_file(url, dest)
        except (urllib.error.URLError, http.client.HTTPException, OSError, DownloadError) as e:
            errors.append(f'  {url}: {e}')
            print(f'  failed: {e}')
    raise DownloadError(f'Could not download {dest.name}:\n' + '\n'.join(errors))


def download_fever_file(filename, dest_dir=RAW_DIR, force=False):
    return download_file([f'{base}/{filename}' for base in FEVER_MIRRORS], Path(dest_dir) / filename, force=force)


def download_claimbuster_file(filename, dest_dir=CLAIMBUSTER_DIR, force=False):
    return download_file([f'{CLAIMBUSTER_BASE}/{filename}?download=1'], Path(dest_dir) / filename, force=force)

# 🟦 TASK 1 · Claim Detection
Splits raw text into sentences (spaCy) and keeps the ones that are **factual claims**, so only those go on to
Task 2 and Task 3. Model: `roberta-base` fine-tuned on ClaimBuster (CFS + UFS = claim, NFS = not a claim).
The decision threshold on P(claim) was tuned on the validation split and is stored in `model/threshold.json`.

### 1.1 Claim detector

In [7]:
SPACY_EXCLUDE = ['ner', 'lemmatizer', 'attribute_ruler', 'tagger']


class ClaimDetector:
    def __init__(self, model_dir, threshold=None, device=None, batch_size=64, max_length=128,
                 spacy_model='en_core_web_sm'):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        model_dir = Path(model_dir)
        self.device = device or get_device()
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(self.device).eval()
        self.batch_size = batch_size
        self.max_length = max_length
        self.spacy_model = spacy_model
        self._nlp = None

        threshold_file = model_dir / 'threshold.json'
        if threshold is None:
            threshold = json.loads(threshold_file.read_text())['threshold'] if threshold_file.exists() else 0.5
        self.threshold = float(threshold)

    @property
    def nlp(self):
        if self._nlp is None:
            import spacy
            self._nlp = spacy.load(self.spacy_model, exclude=SPACY_EXCLUDE)
        return self._nlp

    def split_sentences(self, text):
        return [s.text.strip() for s in self.nlp(text).sents if s.text.strip()]

    @torch.no_grad()
    def score(self, sentences):
        """P(claim) for each sentence."""
        probs = []
        for i in range(0, len(sentences), self.batch_size):
            batch = self.tokenizer(sentences[i:i + self.batch_size], truncation=True, max_length=self.max_length,
                                   padding=True, return_tensors='pt').to(self.device)
            logits = self.model(**batch).logits.float()
            probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
        return np.concatenate(probs) if probs else np.zeros(0)

    def detect(self, text, threshold=None, return_all=False):
        """[{"index", "sentence", "p_claim", "is_claim"}, ...]; only claims unless return_all."""
        threshold = self.threshold if threshold is None else threshold
        sentences = self.split_sentences(text)
        results = [
            {'index': i, 'sentence': s, 'p_claim': float(p), 'is_claim': bool(p >= threshold)}
            for i, (s, p) in enumerate(zip(sentences, self.score(sentences)))
        ]
        return results if return_all else [r for r in results if r['is_claim']]

### 1.2 Training code
Only used when there is no Task 1 model in Drive yet (section 1.3 decides).

**Data: ClaimBuster.** `crowdsourced.csv` → train / val (stratified on the verdict), `groundtruth.csv` → test.
Sentences that also appear in the test set are removed from train/val; duplicate texts are collapsed, and
dropped when their labels disagree.

| Column | Mapping |
|---|---|
| `label` | CFS + UFS = 1, NFS = 0 → main model |
| `label_checkworthy` | CFS = 1, UFS + NFS = 0 → comparison run (original ClaimBuster framing) |

**Training.** Weighted cross-entropy (class weights from the train split). The best step is chosen by validation
macro F1 and saved straight to `model/`; then the threshold is tuned on validation and the test set is scored.

In [8]:
TASK1_LABEL_COLUMNS = ('label', 'label_checkworthy')
TASK1_KEEP_COLUMNS = ['sentence_id', 'text', 'verdict', 'label', 'label_checkworthy', 'speaker', 'file_id']
ID2LABEL = {0: 'non_factual', 1: 'claim'}


# ---------- data ----------
def normalize_sentence(text):
    return ' '.join(str(text).lower().split())


def load_claimbuster(path):
    df = pd.read_csv(path)
    df = df.rename(columns={'Sentence_id': 'sentence_id', 'Text': 'text', 'Verdict': 'verdict',
                            'Speaker': 'speaker', 'File_id': 'file_id'})
    df['text'] = df['text'].astype(str).str.strip()
    df = df[df['text'] != ''].copy()
    df['verdict'] = df['verdict'].astype(int)
    df['label'] = (df['verdict'] >= 0).astype(int)
    df['label_checkworthy'] = (df['verdict'] == 1).astype(int)
    df['norm'] = df['text'].map(normalize_sentence)
    return df


def dedupe(df):
    """Collapse duplicate texts. Drop the text entirely if its copies have different verdicts."""
    verdicts_per_text = df.groupby('norm')['verdict'].nunique()
    conflicting = set(verdicts_per_text[verdicts_per_text > 1].index)
    kept = df[~df['norm'].isin(conflicting)].drop_duplicates('norm', keep='first')
    return kept, len(conflicting), len(df) - len(kept)


def compute_class_weights(labels):
    """n / (n_classes * count_c), so the loss gives both classes equal total weight."""
    counts = labels.value_counts().sort_index()
    return [float(len(labels) / (len(counts) * counts[c])) for c in counts.index]


def build_task1_data(val_size=0.1, seed=42, out_dir=PROCESSED_DIR):
    from sklearn.model_selection import train_test_split

    crowd = load_claimbuster(CLAIMBUSTER_DIR / 'crowdsourced.csv')
    test = load_claimbuster(CLAIMBUSTER_DIR / 'groundtruth.csv')
    stats = {'crowdsourced_raw': len(crowd), 'groundtruth_raw': len(test)}

    in_test = crowd['norm'].isin(set(test['norm']))
    stats['removed_overlap_with_test'] = int(in_test.sum())
    crowd = crowd[~in_test]
    crowd, stats['conflicting_duplicate_texts'], stats['removed_duplicates'] = dedupe(crowd)

    train, val = train_test_split(crowd, test_size=val_size, random_state=seed, stratify=crowd['verdict'])
    splits = {'train': train, 'val': val, 'test': test}
    for name, df in splits.items():
        stats[f'{name}_size'] = len(df)
        stats[f'{name}_verdict_counts'] = {int(k): int(v) for k, v in df['verdict'].value_counts().sort_index().items()}
        for col in TASK1_LABEL_COLUMNS:
            stats[f'{name}_{col}_positive_rate'] = round(float(df[col].mean()), 4)

    meta = {
        'seed': seed,
        'val_size': val_size,
        'label_mapping': {'label': 'CFS(1) + UFS(0) -> 1, NFS(-1) -> 0',
                          'label_checkworthy': 'CFS(1) -> 1, UFS(0) + NFS(-1) -> 0'},
        'class_weights': {col: compute_class_weights(train[col]) for col in TASK1_LABEL_COLUMNS},
        'stats': stats,
    }
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df in splits.items():
        df[TASK1_KEEP_COLUMNS].reset_index(drop=True).to_parquet(out_dir / f'task1_{name}.parquet', index=False)
    (out_dir / 'task1_meta.json').write_text(json.dumps(meta, indent=2))
    print(json.dumps(meta['stats'], indent=2))
    return meta


# ---------- training ----------
class SentenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item['labels'] = int(self.labels[i])
        return item


class WeightedTrainer(transformers.Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


class SaveBestModel(transformers.TrainerCallback):
    """Overwrite <save_dir> with the model whenever the validation metric improves (one copy on disk)."""

    def __init__(self, save_dir, metric='eval_macro_f1'):
        self.save_dir = Path(save_dir)
        self.metric = metric
        self.best_score = -np.inf
        self.best_step = None

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get(self.metric)
        if score is not None and score > self.best_score:
            self.best_score = score
            self.best_step = state.global_step
            model.save_pretrained(self.save_dir)


def softmax_pos(logits):
    return torch.softmax(torch.as_tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


def binary_metrics(labels, probs, threshold=0.5):
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

    preds = (probs >= threshold).astype(int)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'accuracy': accuracy_score(labels, preds),
        'roc_auc': roc_auc_score(labels, probs) if len(set(labels)) > 1 else float('nan'),
        'threshold': float(threshold),
    }


def task1_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    return binary_metrics(np.asarray(labels), softmax_pos(logits))


def tune_threshold(labels, probs):
    """Threshold on P(claim) that maximizes macro F1 on the given (validation) data."""
    from sklearn.metrics import f1_score

    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    scores = [f1_score(labels, (probs >= t).astype(int), average='macro') for t in grid]
    return float(grid[int(np.argmax(scores))])


def load_task1_split(name, label_col, limit=None, seed=42):
    df = pd.read_parquet(PROCESSED_DIR / f'task1_{name}.parquet')
    if limit and limit < len(df):
        df = df.sample(n=limit, random_state=seed)
    return df.reset_index(drop=True), df[label_col].to_numpy()


def train_task1(model_name='roberta-base', label_col='label', epochs=3, lr=2e-5, batch_size=32, grad_accum=1,
                max_length=128, warmup_ratio=0.06, weight_decay=0.01, eval_steps=200, seed=42,
                use_class_weights=True, fp16=True, limit=None, max_steps=-1):
    """Train one Task 1 run. Output in MODELS_DIR/task1_<model>_<label_col>/ (Drive)."""
    config = dict(locals())   # only the arguments: must come before the imports below
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, TrainingArguments

    transformers.set_seed(seed)
    run_name = f"task1_{model_name.split('/')[-1]}_{label_col}"
    out_dir = MODELS_DIR / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    meta = json.loads((PROCESSED_DIR / 'task1_meta.json').read_text())
    class_weights = meta['class_weights'][label_col] if use_class_weights else [1.0, 1.0]
    print(f'Run {run_name} | class weights {class_weights}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()}
    )
    # Some checkpoints (e.g. deberta-v3-small) are stored in fp16; fp16 mixed precision needs fp32 weights.
    model = model.float()

    splits = {name: load_task1_split(name, label_col, limit, seed) for name in ('train', 'val', 'test')}
    datasets = {name: SentenceDataset(df['text'], labels, tokenizer, max_length) for name, (df, labels) in splits.items()}
    print({name: len(ds) for name, ds in datasets.items()})

    steps_per_epoch = int(np.ceil(len(datasets['train']) / (batch_size * grad_accum)))
    total_steps = max_steps if max_steps > 0 else int(steps_per_epoch * epochs)
    eval_steps = min(eval_steps, max(1, total_steps // 2))

    training_args = TrainingArguments(
        output_dir=f'/content/trainer_tmp/{run_name}',   # Trainer scratch files stay off Drive
        num_train_epochs=epochs,
        max_steps=max_steps,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=batch_size * 2,
        warmup_steps=int(warmup_ratio * total_steps),
        weight_decay=weight_decay,
        eval_strategy='steps',
        eval_steps=eval_steps,
        logging_steps=max(1, eval_steps // 4),
        save_strategy='no',
        report_to='none',
        seed=seed,
        fp16=torch.cuda.is_available() and fp16,
        dataloader_pin_memory=torch.cuda.is_available(),
    )
    model_dir = out_dir / 'model'
    keep_best = SaveBestModel(model_dir)
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=datasets['train'],
        eval_dataset=datasets['val'],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=task1_trainer_metrics,
        callbacks=[keep_best],
        class_weights=class_weights,
    )
    trainer.train()

    print(f'Best validation macro F1 {keep_best.best_score:.4f} at step {keep_best.best_step}')
    best = AutoModelForSequenceClassification.from_pretrained(model_dir)
    trainer.model.load_state_dict(best.state_dict())
    del best

    val_labels, test_labels = splits['val'][1], splits['test'][1]
    val_probs = softmax_pos(trainer.predict(datasets['val']).predictions)
    test_probs = softmax_pos(trainer.predict(datasets['test']).predictions)
    threshold = tune_threshold(val_labels, val_probs)

    results = {
        'run': run_name,
        'args': config,
        'class_weights': class_weights,
        'best_step': keep_best.best_step,
        'tuned_threshold': threshold,
        'val@0.5': binary_metrics(val_labels, val_probs, 0.5),
        'val@tuned': binary_metrics(val_labels, val_probs, threshold),
        'test@0.5': binary_metrics(test_labels, test_probs, 0.5),
        'test@tuned': binary_metrics(test_labels, test_probs, threshold),
        'log_history': trainer.state.log_history,
    }
    tokenizer.save_pretrained(model_dir)   # weights were already saved by SaveBestModel
    (model_dir / 'threshold.json').write_text(json.dumps(
        {'threshold': threshold, 'label_col': label_col, 'selected_on': 'val macro F1'}, indent=2))
    (out_dir / 'metrics.json').write_text(json.dumps(results, indent=2, default=float))
    test_df = splits['test'][0][['sentence_id', 'text', 'verdict', label_col]].copy()
    test_df['p_claim'] = test_probs
    test_df['pred'] = (test_probs >= threshold).astype(int)
    test_df.to_csv(out_dir / 'test_predictions.csv', index=False)

    print(f'\nTuned threshold (val): {threshold:.2f}')
    print(f"{'split':<12}{'macroF1':>9}{'prec':>8}{'recall':>8}{'rocAUC':>8}")
    for key in ('val@0.5', 'val@tuned', 'test@0.5', 'test@tuned'):
        m = results[key]
        print(f"{key:<12}{m['macro_f1']:>9.4f}{m['precision']:>8.4f}{m['recall']:>8.4f}{m['roc_auc']:>8.4f}")
    print(f'Saved to {out_dir}')
    return results

### 1.3 Get the Task 1 model from Drive
1. Search every subfolder of `models/` for a model (`model.safetensors` + `config.json`). If several are found,
   prefer one with `threshold.json`, then one with `metrics.json` next to it, then the newest.
2. None found, but `task1_roberta-base_label.zip` is in `DRIVE_DIR` → extract it into `models/` and search again.
3. Still none → download ClaimBuster, build the splits and train (~15 min on a T4), if `TRAIN_TASK1_IF_MISSING` is on.

In [9]:
TRAIN_TASK1_IF_MISSING = True  # @param {type:"boolean"}

TASK1_RUN = 'task1_roberta-base_label'
TASK1_ZIP = DRIVE_DIR / f'{TASK1_RUN}.zip'


def find_task1_model(models_dir=MODELS_DIR):
    """Every folder under models_dir holding a saved classifier, best candidate first."""
    found = [p.parent for p in Path(models_dir).rglob('model.safetensors') if (p.parent / 'config.json').exists()]
    rank = lambda d: ((d / 'threshold.json').exists(), (d.parent / 'metrics.json').exists(),
                      (d / 'model.safetensors').stat().st_mtime)
    return sorted(found, key=rank, reverse=True)


stage('🟦 TASK 1 · get model')
print(f'Searching {MODELS_DIR} ...')
candidates = find_task1_model()
if not candidates and TASK1_ZIP.exists():
    print(f'Extracting {TASK1_ZIP} ...')
    with zipfile.ZipFile(TASK1_ZIP) as zf:
        zf.extractall(DRIVE_DIR)              # the zip contains models/task1_roberta-base_label/...
    candidates = find_task1_model()
if not candidates:
    if not TRAIN_TASK1_IF_MISSING:
        raise FileNotFoundError(f'No Task 1 model under {MODELS_DIR} and TRAIN_TASK1_IF_MISSING is off')
    print('No Task 1 model in Drive: training one (saved straight to Drive) ...')
    for f in CLAIMBUSTER_FILES:
        download_claimbuster_file(f)
    build_task1_data()
    train_task1(model_name='roberta-base', label_col='label', batch_size=32, grad_accum=1)
    candidates = find_task1_model()

for d in candidates:
    print(f"  {'USING' if d == candidates[0] else 'found'}  {d}")
TASK1_MODEL_DIR = candidates[0]
print(sorted(p.name for p in TASK1_MODEL_DIR.iterdir()))
print('threshold.json:', (TASK1_MODEL_DIR / 'threshold.json').read_text())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟦 TASK 1 · get model
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Searching /content/drive/MyDrive/NLP/Praktikum/Final Project/models ...
  USING  /content/drive/MyDrive/NLP/Praktikum/Final Project/models/task1_roberta-base_label/model
['config.json', 'model.safetensors', 'threshold.json', 'tokenizer.json', 'tokenizer_config.json']
threshold.json: {
  "threshold": 0.76,
  "label_col": "label",
  "selected_on": "val macro F1"
}


### 1.4 Load + try it

In [10]:
detector = ClaimDetector(TASK1_MODEL_DIR)
print('device:', detector.device, '| threshold:', detector.threshold)

stage('🟦 TASK 1 · Claim Detection — demo')
demo_text = 'Hello everyone! I think pizza is great. Barack Obama was born in Hawaii in 1961.'
for r in detector.detect(demo_text, return_all=True):
    print(f"{'CLAIM' if r['is_claim'] else '  -  '}  p_claim={r['p_claim']:.3f}  {r['sentence']}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

device: cuda | threshold: 0.76

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟦 TASK 1 · Claim Detection — demo
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  -    p_claim=0.002  Hello everyone!
  -    p_claim=0.002  I think pizza is great.
CLAIM  p_claim=0.999  Barack Obama was born in Hawaii in 1961.


# 🟨 TASK 2 · Evidence Retrieval & Reranking
Finds evidence sentences in Wikipedia for a claim, then reranks them so the most relevant are on top.

```
claim
  -> [1] Document retrieval : BM25 (title + body)                        -> top-5 pages
  -> [2] Sentence retrieval : bi-encoder all-MiniLM-L6-v2                -> top-30 sentences
  -> [3] Reranking          : cross-encoder ms-marco-MiniLM-L-6-v2       -> top-5 sentences
```

### 2.1 FEVER data helpers

In [11]:
PTB_TOKENS = {
    '-LRB-': '(', '-RRB-': ')', '-LSB-': '[', '-RSB-': ']', '-LCB-': '{', '-RCB-': '}',
    '-COLON-': ':', '``': '"', "''": '"',
}


def normalize_page_id(page_id):
    return unicodedata.normalize('NFC', page_id)


def clean_text(text):
    for token, char in PTB_TOKENS.items():
        text = text.replace(token, char)
    text = text.replace('( ', '(').replace(' )', ')')
    return ' '.join(text.split())


def title_from_page_id(page_id):
    return clean_text(page_id.replace('_', ' '))


def load_claims(path, n=None, seed=42, include_nei=False):
    claims = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if row['label'] == 'NOT ENOUGH INFO' and not include_nei:
                continue
            evidence_sets = []
            for ev_set in row['evidence']:
                sentences = []
                for _, _, page_id, line_id in ev_set:
                    if page_id is None:
                        continue
                    sentences.append((normalize_page_id(page_id), int(line_id)))
                sentences = tuple(sorted(set(sentences)))
                if sentences and sentences not in evidence_sets:
                    evidence_sets.append(sentences)
            claims.append({
                'id': row['id'],
                'claim': row['claim'],
                'label': row['label'],
                'evidence_sets': [list(s) for s in evidence_sets],
            })
    if n is not None and n < len(claims):
        claims = random.Random(seed).sample(claims, n)
    return claims


def save_claims(claims, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for c in claims:
            f.write(json.dumps(c, ensure_ascii=False) + '\n')


def read_claims_cache(path):
    claims = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            c = json.loads(line)
            c['evidence_sets'] = [[tuple(s) for s in ev] for ev in c['evidence_sets']]
            claims.append(c)
    return claims


def iter_wiki_pages(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith('.jsonl') and not n.startswith('__MACOSX'))
        for name in names:
            with zf.open(name) as f:
                for raw_line in f:
                    page = json.loads(raw_line)
                    if page['id']:
                        yield page


def parse_lines(lines_field):
    sentences = []
    for row in lines_field.split('\n'):
        parts = row.split('\t')
        if len(parts) < 2 or not parts[1].strip():
            continue
        try:
            line_id = int(parts[0])
        except ValueError:
            continue
        sentences.append((line_id, parts[1]))
    return sentences

### 2.2 BM25 (sparse matrix implementation)

In [12]:
from sklearn.feature_extraction.text import CountVectorizer


class BM25:
    def __init__(self, k1=1.5, b=0.75, stop_words='english'):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(lowercase=True, stop_words=stop_words,
                                          token_pattern=r'(?u)\b\w+\b', dtype=np.float32)
        self.matrix = None
        self.idf = None

    def fit(self, texts):
        tf = self.vectorizer.fit_transform(texts).tocsr()
        n_docs = tf.shape[0]
        doc_len = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = doc_len.mean() if n_docs else 0.0

        df = np.bincount(tf.indices, minlength=tf.shape[1])
        self.idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)

        nnz_per_row = np.diff(tf.indptr)
        len_per_value = np.repeat(doc_len, nnz_per_row)
        denom = tf.data + self.k1 * (1 - self.b + self.b * len_per_value / avgdl)
        tf.data = tf.data * (self.k1 + 1) / denom
        self.matrix = tf.tocsc()
        return self

    def get_scores(self, query):
        q = self.vectorizer.transform([query])
        term_ids = q.indices
        if len(term_ids) == 0:
            return np.zeros(self.matrix.shape[0], dtype=np.float32)
        weights = self.idf[term_ids] * q.data
        return np.asarray(self.matrix[:, term_ids] @ weights).ravel()

    def search(self, query, top_k=10):
        return top_k_indices(self.get_scores(query), top_k)


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return []
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

### 2.3 Retriever — stage [1] document retrieval + stage [2] sentence retrieval

In [13]:
@dataclass
class Evidence:
    page_id: str
    line_id: int
    title: str
    text: str
    doc_score: float = 0.0
    retrieval_score: float = 0.0
    rerank_score: float = None

    @property
    def key(self):
        return (self.page_id, self.line_id)

    def to_dict(self):
        return asdict(self)


def load_corpus(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]


class EvidenceRetriever:
    def __init__(self, pages, bi_encoder_name='sentence-transformers/all-MiniLM-L6-v2', title_weight=2.0, device=None):
        self.pages = pages
        self.title_weight = title_weight
        print(f'Building BM25 index for {len(pages):,} pages ...')
        self.title_index = BM25().fit([p['title'] for p in pages])
        self.body_index = BM25().fit([' '.join(s[1] for s in p['sentences']) for p in pages])
        self.device = device or get_device()
        self.bi_encoder_name = bi_encoder_name
        self._encoder = None

    @property
    def encoder(self):
        if self._encoder is None:
            from sentence_transformers import SentenceTransformer
            self._encoder = SentenceTransformer(self.bi_encoder_name, device=self.device)
        return self._encoder

    # ---------- stage 1: document retrieval ----------
    def retrieve_documents(self, claim, k=5):
        scores = self.title_weight * self.title_index.get_scores(claim) + self.body_index.get_scores(claim)
        return top_k_indices(scores, k)

    # ---------- stage 2: sentence retrieval ----------
    def candidate_sentences(self, claim, doc_k=5):
        """All sentences of the top doc_k pages, unsorted."""
        candidates = []
        for page_idx, doc_score in self.retrieve_documents(claim, doc_k):
            if doc_score <= 0:
                continue
            page = self.pages[page_idx]
            for line_id, text in page['sentences']:
                candidates.append(Evidence(page_id=page['id'], line_id=int(line_id), title=page['title'],
                                           text=text, doc_score=doc_score))
        return candidates

    def score_sentences(self, claim, candidates):
        if not candidates:
            return candidates
        texts = [f'{c.title}: {c.text}' for c in candidates]
        claim_emb = self.encoder.encode([claim], normalize_embeddings=True)
        sent_emb = self.encoder.encode(texts, normalize_embeddings=True, batch_size=64)
        sims = (sent_emb @ claim_emb[0]).astype(float)
        for c, s in zip(candidates, sims):
            c.retrieval_score = float(s)
        return candidates

    def retrieve(self, claim, top_k=30, doc_k=5):
        candidates = self.score_sentences(claim, self.candidate_sentences(claim, doc_k))
        candidates.sort(key=lambda c: c.retrieval_score, reverse=True)
        return candidates[:top_k]

### 2.4 Reranker — stage [3] cross-encoder

In [14]:
class CrossEncoderReranker:
    def __init__(self, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2', device=None, batch_size=32):
        from sentence_transformers import CrossEncoder

        self.model_name = model_name
        self.batch_size = batch_size
        self.model = CrossEncoder(model_name, device=device or get_device())

    def rerank(self, claim, candidates, top_k=5):
        if not candidates:
            return []
        pairs = [(claim, f'{c.title}: {c.text}') for c in candidates]
        scores = self.model.predict(pairs, batch_size=self.batch_size, show_progress_bar=False)
        for c, s in zip(candidates, scores):
            c.rerank_score = float(s)
        return sorted(candidates, key=lambda c: c.rerank_score, reverse=True)[:top_k]

### 2.5 Evidence pipeline (stages 1 → 2 → 3)

In [15]:
CORPUS_PATH = PROCESSED_DIR / 'wiki_subset.jsonl'
CLAIMS_SAMPLE_PATH = PROCESSED_DIR / 'claims_sample.jsonl'


class EvidencePipeline:
    def __init__(self, retriever, reranker=None, doc_k=5, n_candidates=30):
        self.retriever = retriever
        self.reranker = reranker
        self.doc_k = doc_k
        self.n_candidates = n_candidates

    @classmethod
    def from_corpus_file(cls, corpus_path=CORPUS_PATH, use_reranker=True, **kwargs):
        retriever = EvidenceRetriever(load_corpus(corpus_path))
        reranker = CrossEncoderReranker() if use_reranker else None
        return cls(retriever, reranker, **kwargs)

    def run(self, claim, top_k=5, as_dict=True):
        candidates = self.retriever.retrieve(claim, top_k=self.n_candidates, doc_k=self.doc_k)
        results = self.reranker.rerank(claim, candidates, top_k=top_k) if self.reranker else candidates[:top_k]
        return [r.to_dict() for r in results] if as_dict else results

### 2.6 Build the Wikipedia corpus subset (once, saved to Drive)
Indexing all 5.4M pages is too heavy, so the corpus is a subset of:
1. **gold pages**: evidence pages of the 500 sampled claims
2. **hard negatives**: pages whose title appears in a claim (e.g. "Soul Food" also pulls "Soul Food (TV series)")
3. **random pages**: noise

Recall on this subset is **more optimistic** than on full Wikipedia, and a claim outside the sample will
usually not find its evidence. `NOT ENOUGH INFO` claims are not sampled (they have no evidence).

In [16]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

TOTAL_WIKI_PAGES = 5_416_537


def claim_ngrams(claims, max_n=3):
    grams = set()
    for c in claims:
        tokens = re.findall(r'\w+', c['claim'].lower())
        for n in range(1, max_n + 1):
            for i in range(len(tokens) - n + 1):
                gram = tokens[i:i + n]
                if n == 1 and (gram[0] in ENGLISH_STOP_WORDS or len(gram[0]) < 3):
                    continue
                grams.add(' '.join(gram))
    return grams


def base_title(title):
    title = re.sub(r'\s*\(.*\)\s*$', '', title)
    return ' '.join(re.findall(r'\w+', title.lower()))


def build_subset_corpus(claims, zip_path, n_random=20000, seed=42, max_sentences=None):
    gold_pages = {page for c in claims for ev in c['evidence_sets'] for page, _ in ev}
    grams = claim_ngrams(claims)
    rng = random.Random(seed)
    random_prob = n_random / TOTAL_WIKI_PAGES

    corpus = []
    stats = {'gold': 0, 'hard_negative': 0, 'random': 0}
    for page in tqdm(iter_wiki_pages(zip_path), total=TOTAL_WIKI_PAGES, desc='Scan wiki'):
        page_id = normalize_page_id(page['id'])
        title = title_from_page_id(page_id)
        if page_id in gold_pages:
            source = 'gold'
        elif base_title(title) in grams:
            source = 'hard_negative'
        elif rng.random() < random_prob:
            source = 'random'
        else:
            continue

        sentences = parse_lines(page['lines'])
        if max_sentences:
            sentences = sentences[:max_sentences]
        if not sentences:
            continue
        corpus.append({
            'id': page_id,
            'title': title,
            'sentences': [[line_id, clean_text(text)] for line_id, text in sentences],
            'source': source,
        })
        stats[source] += 1

    missing = gold_pages - {p['id'] for p in corpus if p['source'] == 'gold'}
    return corpus, stats, missing


def save_corpus(corpus, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for page in corpus:
            f.write(json.dumps(page, ensure_ascii=False) + '\n')


def build_corpus(split='shared_task_dev.jsonl', n_claims=500, n_random=20000, seed=42):
    claims_path = download_fever_file(split)
    zip_path = download_fever_file('wiki-pages.zip')          # 1.7 GB, skipped if already in Drive
    claims = load_claims(claims_path, n=n_claims, seed=seed, include_nei=False)
    print(f'Sampled claims: {len(claims)}')

    corpus, stats, missing = build_subset_corpus(claims, zip_path, n_random=n_random, seed=seed)
    print(f'Pages in corpus: {len(corpus):,} -> {stats}')
    if missing:
        print(f'Warning: {len(missing)} gold pages not found in the dump, e.g. {list(missing)[:5]}')
    save_claims(claims, CLAIMS_SAMPLE_PATH)
    save_corpus(corpus, CORPUS_PATH)
    print(f'Saved to {PROCESSED_DIR}')


stage('🟨 TASK 2 · corpus')
if CORPUS_PATH.exists() and CLAIMS_SAMPLE_PATH.exists():
    print(f'Corpus already in Drive, skipping: {CORPUS_PATH}')
else:
    build_corpus(n_claims=500, n_random=20000)
!ls -lh "{PROCESSED_DIR}"


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟨 TASK 2 · corpus
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Download https://fever.ai/download/fever/shared_task_dev.jsonl -> /content/drive/MyDrive/NLP/Praktikum/Final Project/data/raw/shared_task_dev.jsonl


shared_task_dev.jsonl:   0%|          | 0.00/4.35M [00:00<?, ?B/s]

Download https://fever.ai/download/fever/wiki-pages.zip -> /content/drive/MyDrive/NLP/Praktikum/Final Project/data/raw/wiki-pages.zip


wiki-pages.zip:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Sampled claims: 500


Scan wiki:   0%|          | 0/5416537 [00:00<?, ?it/s]

Pages in corpus: 30,997 -> {'gold': 476, 'hard_negative': 10808, 'random': 19713}
Saved to /content/drive/MyDrive/NLP/Praktikum/Final Project/data/processed
total 23M
-rw------- 1 root root  77K Nov 21 15:34 claims_sample.jsonl
-rw------- 1 root root 1.2K Nov 21 15:02 task1_meta.json
-rw------- 1 root root  69K Nov 21 15:02 task1_test.parquet
-rw------- 1 root root 1.4M Nov 21 15:02 task1_train.parquet
-rw------- 1 root root 159K Nov 21 15:02 task1_val.parquet
-rw------- 1 root root  22M Nov 21 15:34 wiki_subset.jsonl


### 2.7 Load + try it
The demo uses a claim from `claims_sample.jsonl`, because the corpus only covers those claims.

In [17]:
evidence_pipe = EvidencePipeline.from_corpus_file(CORPUS_PATH)
sample_claims = read_claims_cache(CLAIMS_SAMPLE_PATH)
print(f'{len(evidence_pipe.retriever.pages):,} pages in corpus, {len(sample_claims)} sample claims')

DEMO_CLAIM = sample_claims[0]
stage('🟨 TASK 2 · Evidence Retrieval — demo')
print(f"Claim: {DEMO_CLAIM['claim']}   (FEVER label: {DEMO_CLAIM['label']})\n")
demo_evidence = evidence_pipe.run(DEMO_CLAIM['claim'], top_k=5)
for i, ev in enumerate(demo_evidence, 1):
    print(f"{i}. [{ev['title']} #{ev['line_id']}] rerank={ev['rerank_score']:.2f}\n   {ev['text']}")

Building BM25 index for 30,997 pages ...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

... (14 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1. [South Island #5] rerank=8.89
   It has a 32 percent larger landmass than the North Island so is sometimes referred to as the " mainland " of New Zealand , especially by South Island residents , but only 23 percent of New Zealand 's million inhabitants live there .
2. [South Island (South Australia) #0] rerank=5.17
   South Island (also called Peter Island , South Islet and West Bay Island) is an island located in the Australian state of South Australia at the south end of Pondalowie Bay on the south-west extremity of Yorke Peninsula about 11 km north-west by west of the town of Stenhouse Bay .
3. [South Island (disambiguation) #3] rerank=4.51
   South Island may also refer to :
4. [South Island #0] rerank=4.50
   The South Island or Te Waipounamu (Māori) is the larger of the two major islands of New Zealand , the other being the smaller but more populous North Island .
5. [South Island (disambiguation) #0] rerank=4.32
   The South Island is the southern of the two main islands of N

### 2.8 Evaluation (optional) → `outputs/task2/`
- **Doc recall@k**: every page of some gold evidence set is in the top-k BM25 pages
- **Candidate recall**: a complete gold set is among the 30 candidates (upper bound for the reranker)
- **Evidence recall@5**: official FEVER metric, a complete gold set is in the top-5 sentences
- **Hit@1**: the first sentence is gold evidence; **MRR**: mean 1/rank of the first gold sentence

In [18]:
def full_set_found(evidence_sets, found_keys):
    found_keys = set(found_keys)
    return any(all(tuple(s) in found_keys for s in ev) for ev in evidence_sets)


def doc_set_found(evidence_sets, found_pages):
    found_pages = set(found_pages)
    return any(all(page in found_pages for page, _ in ev) for ev in evidence_sets)


def reciprocal_rank(ranked_keys, gold_keys):
    for rank, key in enumerate(ranked_keys, 1):
        if key in gold_keys:
            return 1.0 / rank
    return 0.0


def evaluate_retrieval(claims, retriever, reranker=None, doc_k=5, n_candidates=30, top_k=5, doc_ks=(1, 5, 10)):
    totals, predictions = {}, []

    def add(name, value):
        totals[name] = totals.get(name, 0.0) + float(value)

    for c in tqdm(claims, desc='Evaluate retrieval'):
        gold_sets = c['evidence_sets']
        gold_keys = {tuple(s) for ev in gold_sets for s in ev}

        # --- document retrieval ---
        docs = retriever.retrieve_documents(c['claim'], k=max(doc_ks))
        doc_ids = [retriever.pages[i]['id'] for i, _ in docs]
        for k in doc_ks:
            add(f'doc_recall@{k}', doc_set_found(gold_sets, doc_ids[:k]))

        # --- sentence retrieval (bi-encoder) ---
        candidates = retriever.retrieve(c['claim'], top_k=n_candidates, doc_k=doc_k)
        before = [x.key for x in candidates]
        add('candidate_recall', full_set_found(gold_sets, before))
        add('before/evidence_recall@5', full_set_found(gold_sets, before[:top_k]))
        add('before/hit@1', bool(before) and before[0] in gold_keys)
        add('before/mrr', reciprocal_rank(before, gold_keys))

        # --- reranking (cross-encoder) ---
        final = candidates[:top_k]
        if reranker is not None:
            reranked = reranker.rerank(c['claim'], candidates, top_k=len(candidates))
            after = [x.key for x in reranked]
            add('after/evidence_recall@5', full_set_found(gold_sets, after[:top_k]))
            add('after/hit@1', bool(after) and after[0] in gold_keys)
            add('after/mrr', reciprocal_rank(after, gold_keys))
            final = reranked[:top_k]

        predictions.append({'id': c['id'], 'claim': c['claim'], 'label': c['label'], 'gold_evidence': gold_sets,
                            'predicted_evidence': [x.to_dict() for x in final]})

    return {k: v / len(claims) for k, v in totals.items()}, predictions


def print_retrieval_metrics(metrics):
    print('\n=== Document retrieval (BM25) ===')
    for k, v in metrics.items():
        if k.startswith('doc_recall'):
            print(f'  {k:<22} {v:.4f}')
    print(f"  {'candidate_recall':<22} {metrics['candidate_recall']:.4f}  (upper bound for the reranker)")
    print('\n=== Sentence retrieval ===')
    print(f"  {'metric':<20} {'before rerank':>15} {'after rerank':>15}")
    for name in ['evidence_recall@5', 'hit@1', 'mrr']:
        before, after = metrics.get(f'before/{name}'), metrics.get(f'after/{name}')
        after_str = f'{after:.4f}' if after is not None else '-'
        print(f'  {name:<20} {before:>15.4f} {after_str:>15}')

In [19]:
RUN_TASK2_EVAL = True  # @param {type:"boolean"}
TASK2_EVAL_LIMIT = 0  # @param {type:"integer"}
# TASK2_EVAL_LIMIT = 0 -> all 500 claims; e.g. 50 for a quick test

if RUN_TASK2_EVAL:
    stage('🟨 TASK 2 · Evaluation')
    eval_claims = read_claims_cache(CLAIMS_SAMPLE_PATH)
    if TASK2_EVAL_LIMIT:
        eval_claims = eval_claims[:TASK2_EVAL_LIMIT]
    print(f'Claims: {len(eval_claims)}')

    start = time.time()
    task2_metrics, task2_predictions = evaluate_retrieval(
        eval_claims, evidence_pipe.retriever, evidence_pipe.reranker,
        doc_k=evidence_pipe.doc_k, n_candidates=evidence_pipe.n_candidates, top_k=5)
    elapsed = time.time() - start
    print_retrieval_metrics(task2_metrics)
    print(f'\nTime: {elapsed:.1f} s ({elapsed / len(eval_claims):.2f} s/claim)')

    out_dir = OUTPUTS_DIR / 'task2'
    out_dir.mkdir(parents=True, exist_ok=True)
    config = {'n_claims': len(eval_claims), 'doc_k': evidence_pipe.doc_k,
              'n_candidates': evidence_pipe.n_candidates, 'top_k': 5}
    (out_dir / 'metrics.json').write_text(json.dumps({**config, **task2_metrics}, indent=2))
    with open(out_dir / 'predictions.jsonl', 'w', encoding='utf-8') as f:
        for p in task2_predictions:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    print(f'Saved to {out_dir}')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟨 TASK 2 · Evaluation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Claims: 500


Evaluate retrieval:   0%|          | 0/500 [00:00<?, ?it/s]


=== Document retrieval (BM25) ===
  doc_recall@1           0.6460
  doc_recall@5           0.9000
  doc_recall@10          0.9220
  candidate_recall       0.9000  (upper bound for the reranker)

=== Sentence retrieval ===
  metric                 before rerank    after rerank
  evidence_recall@5             0.8160          0.8840
  hit@1                         0.5680          0.7960
  mrr                           0.7057          0.8631

Time: 34.4 s (0.07 s/claim)
Saved to /content/drive/MyDrive/NLP/Praktikum/Final Project/outputs/task2


# 🟥 TASK 3 · Verification (NLI)
Takes a claim and its top-k evidence sentences from Task 2 and decides **SUPPORTS / REFUTES / NOT ENOUGH INFO**.

- Model: `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`, zero-shot (no extra training).
- Premise = `"<page title>: <sentence>"`, hypothesis = claim. Every sentence is scored on its own, plus **one extra
  pair with all top-k sentences concatenated**, for claims that need two sentences together (multi-hop).
- Aggregation: `support = max P(entailment)`, `refute = max P(contradiction)` over all pairs;
  `max(support, refute) < threshold` → NOT ENOUGH INFO, otherwise the larger one wins.

### 3.1 NLI verifier

In [20]:
NLI_MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
LABELS = ('SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO')
NLI_KEYS = ('entailment', 'neutral', 'contradiction')


def format_premise(ev):
    """Evidence dict (or plain string) -> premise text."""
    if isinstance(ev, str):
        return clean_text(ev)
    text = clean_text(ev['text'])
    title = ev.get('title')
    return f'{title}: {text}' if title else text


def aggregate(sentence_probs, concat_probs=None, threshold=0.5):
    """NLI probabilities -> (FEVER label, scores per label).

    sentence_probs: (n_evidence, 3) in NLI_KEYS order; concat_probs: (3,) or None.
    """
    rows = [np.asarray(p, dtype=float) for p in sentence_probs]
    if concat_probs is not None:
        rows.append(np.asarray(concat_probs, dtype=float))
    if not rows:
        return 'NOT ENOUGH INFO', {'SUPPORTS': 0.0, 'REFUTES': 0.0, 'NOT ENOUGH INFO': 1.0}

    probs = np.stack(rows)
    support = float(probs[:, 0].max())
    refute = float(probs[:, 2].max())
    scores = {'SUPPORTS': support, 'REFUTES': refute, 'NOT ENOUGH INFO': 1.0 - max(support, refute)}
    if max(support, refute) < threshold:
        return 'NOT ENOUGH INFO', scores
    return ('SUPPORTS' if support >= refute else 'REFUTES'), scores


class NLIVerifier:
    def __init__(self, model_name=NLI_MODEL_NAME, threshold=0.5, use_concat=True, device=None,
                 batch_size=16, max_length=512):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.model_name = model_name
        self.threshold = float(threshold)
        self.use_concat = use_concat
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = device or get_device()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device).eval()

        # Label order differs between NLI checkpoints, so read it from the config.
        id2label = {i: name.lower() for i, name in self.model.config.id2label.items()}
        self.column_order = []
        for key in NLI_KEYS:
            matches = [i for i, name in id2label.items() if name.startswith(key[:7])]
            if len(matches) != 1:
                raise ValueError(f"{model_name}: cannot find '{key}' in id2label {self.model.config.id2label}")
            self.column_order.append(matches[0])

    @torch.no_grad()
    def predict_pairs(self, premises, hypotheses):
        """NLI probabilities, shape (n, 3) in NLI_KEYS order."""
        out = []
        for i in range(0, len(premises), self.batch_size):
            batch = self.tokenizer(
                premises[i:i + self.batch_size], hypotheses[i:i + self.batch_size],
                truncation='only_first', max_length=self.max_length, padding=True, return_tensors='pt',
            ).to(self.device)
            logits = self.model(**batch).logits.float()
            out.append(torch.softmax(logits, dim=-1)[:, self.column_order].cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, 3))

    def score(self, claim, evidence):
        """Raw NLI probabilities: (sentence_probs (n, 3), concat_probs (3,) or None)."""
        premises = [format_premise(ev) for ev in evidence]
        if not premises:
            return np.zeros((0, 3)), None
        pairs = list(premises)
        if self.use_concat and len(premises) > 1:
            pairs.append(' '.join(premises))
        probs = self.predict_pairs(pairs, [claim] * len(pairs))
        if len(pairs) > len(premises):
            return probs[:-1], probs[-1]
        return probs, None

    def verify(self, claim, evidence, threshold=None):
        """Verdict for one claim given its evidence (list of evidence dicts or strings)."""
        threshold = self.threshold if threshold is None else threshold
        sentence_probs, concat_probs = self.score(claim, evidence)
        label, scores = aggregate(sentence_probs, concat_probs, threshold)

        scored_evidence = []
        for ev, p in zip(evidence, sentence_probs):
            ev = {'text': ev} if isinstance(ev, str) else dict(ev)
            ev['nli'] = {k: float(v) for k, v in zip(NLI_KEYS, p)}
            ev['nli_label'] = NLI_KEYS[int(np.argmax(p))]
            scored_evidence.append(ev)
        return {
            'claim': claim,
            'label': label,
            'confidence': scores[label],
            'scores': scores,
            'concat_nli': None if concat_probs is None else {k: float(v) for k, v in zip(NLI_KEYS, concat_probs)},
            'evidence': scored_evidence,
        }

### 3.2 Load + try it (on the Task 2 demo evidence)

In [21]:
verifier = NLIVerifier()   # threshold 0.5
print('device:', verifier.device)

stage('🟥 TASK 3 · Verification — demo')
res = verifier.verify(DEMO_CLAIM['claim'], demo_evidence)
print(f"Claim:   {DEMO_CLAIM['claim']}")
print(f"Verdict: {res['label']} ({res['confidence']:.3f})   gold: {DEMO_CLAIM['label']}\n")
for i, ev in enumerate(res['evidence'], 1):
    nli = ev['nli']
    print(f"{i}. E={nli['entailment']:.2f} N={nli['neutral']:.2f} C={nli['contradiction']:.2f}  [{ev['title']}] {ev['text']}")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

device: cuda

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟥 TASK 3 · Verification — demo
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Claim:   South Island is referred to as the "mainland" by South Island residents.
Verdict: SUPPORTS (0.991)   gold: SUPPORTS

1. E=0.98 N=0.02 C=0.00  [South Island] It has a 32 percent larger landmass than the North Island so is sometimes referred to as the " mainland " of New Zealand , especially by South Island residents , but only 23 percent of New Zealand 's million inhabitants live there .
2. E=0.01 N=0.92 C=0.07  [South Island (South Australia)] South Island (also called Peter Island , South Islet and West Bay Island) is an island located in the Australian state of South Australia at the south end of Pondalowie Bay on the south-west extremity of Yorke Peninsula about 11 km north-west by west of the town of Stenhouse Bay .
3. E=0.00 N=0.99 C=0.00  [South 

### 3.3 Evaluation (optional) → `outputs/task3/`
Claims: the 500 SUPPORTS/REFUTES claims from `claims_sample.jsonl` + 250 NOT ENOUGH INFO claims from `shared_task_dev`.
NLI probabilities are computed once per claim, then the NEI threshold is swept for free: it is tuned on one half
of the claims and reported on the other half.

- `retrieved`: evidence from Task 2 (the real pipeline)
- `oracle`: gold evidence for SUPPORTS/REFUTES claims (upper bound for the NLI step)

Metrics: accuracy, macro F1, per-class P/R/F1, confusion matrix, and **FEVER score**
(label correct AND, for SUPPORTS/REFUTES, a complete gold evidence set in the top-k evidence).

In [22]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

NLI_THRESHOLDS = [round(t, 2) for t in np.arange(0.30, 0.96, 0.05)]


def load_eval_claims(n_nei=250, seed=42):
    claims = read_claims_cache(CLAIMS_SAMPLE_PATH)
    if n_nei:
        dev_path = download_fever_file('shared_task_dev.jsonl')
        nei = [c for c in load_claims(dev_path, include_nei=True) if c['label'] == 'NOT ENOUGH INFO']
        claims += random.Random(seed).sample(nei, min(n_nei, len(nei)))
    return claims


def gold_evidence(claim, sentence_lookup):
    """First gold evidence set whose sentences are all in the corpus, as evidence dicts."""
    for ev_set in claim['evidence_sets']:
        keys = [tuple(k) for k in ev_set]
        if all(k in sentence_lookup for k in keys):
            return [{'page_id': p, 'line_id': l, **sentence_lookup[(p, l)]} for p, l in keys]
    return None


def nli_metrics(rows, threshold):
    y_true, y_pred, fever = [], [], []
    for r in rows:
        label, _ = aggregate(r['sentence_probs'], r['concat_probs'], threshold)
        y_true.append(r['label'])
        y_pred.append(label)
        evidence_ok = r['label'] == 'NOT ENOUGH INFO' or full_set_found(r['gold_evidence'], r['evidence_keys'])
        fever.append(label == r['label'] and evidence_ok)

    p, rc, f, support = precision_recall_fscore_support(y_true, y_pred, labels=list(LABELS), zero_division=0)
    return {
        'threshold': threshold,
        'n': len(rows),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=list(LABELS), average='macro', zero_division=0),
        'fever_score': float(np.mean(fever)),
        'per_class': {lab: {'precision': float(p[i]), 'recall': float(rc[i]), 'f1': float(f[i]), 'support': int(support[i])}
                      for i, lab in enumerate(LABELS)},
        'confusion_matrix': {'labels': list(LABELS),
                             'matrix': confusion_matrix(y_true, y_pred, labels=list(LABELS)).tolist()},
    }


def print_nli_metrics(name, m):
    print(f"\n=== {name} (n={m['n']}, threshold={m['threshold']:.2f}) ===")
    print(f"  accuracy    {m['accuracy']:.4f}")
    print(f"  macro F1    {m['macro_f1']:.4f}")
    print(f"  FEVER score {m['fever_score']:.4f}")
    print(f"  {'class':<16} {'P':>6} {'R':>6} {'F1':>6} {'n':>5}")
    for lab, s in m['per_class'].items():
        print(f"  {lab:<16} {s['precision']:>6.3f} {s['recall']:>6.3f} {s['f1']:>6.3f} {s['support']:>5}")
    print('  confusion (rows = gold, cols = predicted S / R / NEI):')
    for lab, row in zip(LABELS, m['confusion_matrix']['matrix']):
        print(f'    {lab:<16} {row}')


def evaluate_verification(evidence_pipe, verifier, oracle_evidence=False, n_nei=250, limit=None, seed=42, top_k=5):
    """Runs Task 2 + Task 3 on the eval claims, prints metrics, saves them to OUTPUTS_DIR/task3."""
    config = {'oracle_evidence': oracle_evidence, 'n_nei': n_nei, 'limit': limit, 'seed': seed, 'top_k': top_k,
              'model': verifier.model_name, 'threshold': verifier.threshold, 'use_concat': verifier.use_concat}
    claims = load_eval_claims(n_nei, seed)
    random.Random(seed).shuffle(claims)
    if limit:
        claims = claims[:limit]
    print(f'Claims: {len(claims)}  ' + ', '.join(f"{lab}={sum(c['label'] == lab for c in claims)}" for lab in LABELS))

    sentence_lookup = {}
    if oracle_evidence:
        for page in evidence_pipe.retriever.pages:
            for line_id, text in page['sentences']:
                sentence_lookup[(page['id'], int(line_id))] = {'title': page['title'], 'text': text}

    rows, n_oracle_missing = [], 0
    start = time.time()
    for c in tqdm(claims, desc='NLI'):
        evidence = evidence_pipe.run(c['claim'], top_k=top_k)
        nli_evidence = evidence
        if oracle_evidence and c['label'] != 'NOT ENOUGH INFO':
            gold = gold_evidence(c, sentence_lookup)
            if gold is None:
                n_oracle_missing += 1
            else:
                nli_evidence = gold
        sentence_probs, concat_probs = verifier.score(c['claim'], nli_evidence)
        rows.append({
            'id': c['id'],
            'claim': c['claim'],
            'label': c['label'],
            'gold_evidence': c['evidence_sets'],
            'evidence_keys': [(e['page_id'], e['line_id']) for e in nli_evidence],
            'evidence': [{'page_id': e['page_id'], 'line_id': e['line_id'], 'title': e['title'], 'text': e['text']}
                         for e in nli_evidence],
            'sentence_probs': sentence_probs.tolist(),
            'concat_probs': None if concat_probs is None else concat_probs.tolist(),
        })
    elapsed = time.time() - start
    if oracle_evidence:
        print(f'Oracle: {n_oracle_missing} S/R claims have no gold set fully in the corpus, used retrieval instead')

    # --- fixed threshold on all claims ---
    fixed = nli_metrics(rows, verifier.threshold)
    print_nli_metrics('All claims, fixed threshold', fixed)

    # --- tune threshold on one half, report on the other ---
    half = len(rows) // 2
    tune_rows, test_rows = rows[:half], rows[half:]
    sweep = [nli_metrics(tune_rows, t) for t in NLI_THRESHOLDS]
    best = max(sweep, key=lambda m: m['macro_f1'])
    print('\nThreshold sweep on the tune half (macro F1 / FEVER score):')
    for m in sweep:
        mark = '  <- best' if m is best else ''
        print(f"  {m['threshold']:.2f}  {m['macro_f1']:.4f}  {m['fever_score']:.4f}{mark}")
    tuned = nli_metrics(test_rows, best['threshold'])
    print_nli_metrics('Test half, tuned threshold', tuned)
    print(f'\nTime: {elapsed:.1f} s ({elapsed / max(len(rows), 1):.2f} s/claim)')

    out_dir = OUTPUTS_DIR / 'task3'
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = 'oracle' if oracle_evidence else 'retrieved'
    (out_dir / f'metrics_{tag}.json').write_text(json.dumps({
        'args': config, 'elapsed_s': elapsed, 'fixed_threshold': fixed,
        'tuned_threshold': best['threshold'], 'tuned_test_half': tuned,
        'sweep_tune_half': [{k: m[k] for k in ('threshold', 'accuracy', 'macro_f1', 'fever_score')} for m in sweep],
    }, indent=2))
    with open(out_dir / f'predictions_{tag}.jsonl', 'w', encoding='utf-8') as f:
        for r in rows:
            label, scores = aggregate(r['sentence_probs'], r['concat_probs'], verifier.threshold)
            f.write(json.dumps({**r, 'predicted': label, 'scores': scores}, ensure_ascii=False) + '\n')
    print(f'Saved to {out_dir}')
    return fixed, tuned

In [23]:
RUN_TASK3_EVAL_RETRIEVED = True  # @param {type:"boolean"}
RUN_TASK3_EVAL_ORACLE = True  # @param {type:"boolean"}
TASK3_EVAL_LIMIT = 0  # @param {type:"integer"}
# TASK3_EVAL_LIMIT = 0 -> all 750 claims; e.g. 50 for a quick test

if RUN_TASK3_EVAL_RETRIEVED:
    stage('🟥 TASK 3 · Evaluation — retrieved evidence (real pipeline)')
    evaluate_verification(evidence_pipe, verifier, oracle_evidence=False, limit=TASK3_EVAL_LIMIT or None)

if RUN_TASK3_EVAL_ORACLE:
    stage('🟥 TASK 3 · Evaluation — oracle (gold) evidence')
    evaluate_verification(evidence_pipe, verifier, oracle_evidence=True, limit=TASK3_EVAL_LIMIT or None)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟥 TASK 3 · Evaluation — retrieved evidence (real pipeline)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Claims: 750  SUPPORTS=230, REFUTES=270, NOT ENOUGH INFO=250


NLI:   0%|          | 0/750 [00:00<?, ?it/s]


=== All claims, fixed threshold (n=750, threshold=0.50) ===
  accuracy    0.6827
  macro F1    0.6357
  FEVER score 0.6240
  class                 P      R     F1     n
  SUPPORTS          0.679  0.939  0.788   230
  REFUTES           0.649  0.878  0.746   270
  NOT ENOUGH INFO   0.881  0.236  0.372   250
  confusion (rows = gold, cols = predicted S / R / NEI):
    SUPPORTS         [216, 13, 1]
    REFUTES          [26, 237, 7]
    NOT ENOUGH INFO  [76, 115, 59]

Threshold sweep on the tune half (macro F1 / FEVER score):
  0.30  0.5849  0.5893
  0.35  0.5901  0.5920
  0.40  0.6147  0.6053
  0.45  0.6262  0.6133
  0.50  0.6437  0.6240
  0.55  0.6693  0.6400
  0.60  0.6855  0.6507
  0.65  0.6947  0.6560
  0.70  0.7021  0.6613
  0.75  0.7060  0.6640
  0.80  0.7222  0.6747
  0.85  0.7473  0.6933
  0.90  0.7683  0.7120
  0.95  0.7818  0.7280  <- best

=== Test half, tuned threshold (n=375, threshold=0.95) ===
  accuracy    0.7707
  macro F1    0.7710
  FEVER score 0.7413
  class           

NLI:   0%|          | 0/750 [00:00<?, ?it/s]

Oracle: 0 S/R claims have no gold set fully in the corpus, used retrieval instead

=== All claims, fixed threshold (n=750, threshold=0.50) ===
  accuracy    0.6640
  macro F1    0.6239
  FEVER score 0.6640
  class                 P      R     F1     n
  SUPPORTS          0.712  0.935  0.808   230
  REFUTES           0.655  0.830  0.732   270
  NOT ENOUGH INFO   0.557  0.236  0.331   250
  confusion (rows = gold, cols = predicted S / R / NEI):
    SUPPORTS         [215, 3, 12]
    REFUTES          [11, 224, 35]
    NOT ENOUGH INFO  [76, 115, 59]

Threshold sweep on the tune half (macro F1 / FEVER score):
  0.30  0.5809  0.6453
  0.35  0.5857  0.6480
  0.40  0.6027  0.6533
  0.45  0.6111  0.6560
  0.50  0.6208  0.6587
  0.55  0.6316  0.6587
  0.60  0.6420  0.6640
  0.65  0.6479  0.6667
  0.70  0.6524  0.6693
  0.75  0.6456  0.6587
  0.80  0.6510  0.6587
  0.85  0.6674  0.6693  <- best
  0.90  0.6645  0.6613
  0.95  0.5910  0.5920

=== Test half, tuned threshold (n=375, threshold=0.85) ==

# 🔗 FULL PIPELINE · Task 1 → Task 2 → Task 3
Type any English text in the form. Leave it empty to use a demo text built around a FEVER claim that is in the corpus.

Keep in mind the corpus is a subset: a claim about a topic outside the 500 sampled claims will usually get
unrelated evidence, and then a NOT ENOUGH INFO verdict. The result is also saved to `outputs/pipeline/last_run.json`.

In [24]:
class FactCheckPipeline:
    """raw text -> claims (Task 1) -> evidence (Task 2) -> verdict (Task 3)."""

    def __init__(self, claim_detector=None, evidence_pipeline=None, verifier=None, top_k=5):
        self.claim_detector = claim_detector
        self.evidence_pipeline = evidence_pipeline
        self.verifier = verifier
        self.top_k = top_k

    def verify_claim(self, claim, check_worthy_confidence=None):
        evidence = self.evidence_pipeline.run(claim, top_k=self.top_k)
        result = self.verifier.verify(claim, evidence)
        return {
            'claim': claim,
            'check_worthy_confidence': check_worthy_confidence,
            'verdict': result['label'],
            'verdict_confidence': result['confidence'],
            'scores': result['scores'],
            'evidence': result['evidence'],
        }

    def check(self, text):
        """Every claim found in `text` through retrieval + NLI. Without a detector, the text is one claim."""
        if self.claim_detector is None:
            claims = [{'sentence': text.strip(), 'p_claim': None}]
        else:
            claims = self.claim_detector.detect(text)
        return [self.verify_claim(c['sentence'], c['p_claim']) for c in claims]


pipeline = FactCheckPipeline(claim_detector=detector, evidence_pipeline=evidence_pipe, verifier=verifier, top_k=5)

In [25]:
TEXT = ''  # @param {type:"string"}
TOP_K = 5  # @param {type:"integer"}

pipeline.top_k = TOP_K
text = TEXT.strip() or f"Good evening everyone, thanks for coming. {sample_claims[1]['claim']} I really love pizza here."
print('Input:', text)

stage('🟦 TASK 1 · Claim Detection')
sentences = detector.detect(text, return_all=True)
for r in sentences:
    print(f"{'CLAIM' if r['is_claim'] else '  -  '}  p_claim={r['p_claim']:.3f}  {r['sentence']}")
claims = [r for r in sentences if r['is_claim']]
if not claims:
    print('\nNo factual claims found, nothing to verify.')

results = []
for n, c in enumerate(claims, 1):
    r = pipeline.verify_claim(c['sentence'], c['p_claim'])   # Task 2 + Task 3
    results.append(r)

    stage(f"🟨 TASK 2 · Evidence  —  claim {n}/{len(claims)}: {r['claim']}")
    for i, ev in enumerate(r['evidence'], 1):
        print(f"{i}. [{ev['title']} #{ev['line_id']}] rerank={ev['rerank_score']:.2f}  {ev['text']}")

    stage(f'🟥 TASK 3 · Verdict  —  claim {n}/{len(claims)}')
    print(f"{r['verdict']}  (confidence {r['verdict_confidence']:.3f})")
    for i, ev in enumerate(r['evidence'], 1):
        print(f"  {i}. E={ev['nli']['entailment']:.2f} C={ev['nli']['contradiction']:.2f}  {ev['title']}")

out_path = OUTPUTS_DIR / 'pipeline' / 'last_run.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({'text': text, 'results': results}, indent=2, ensure_ascii=False, default=float))
print(f'\nSaved to {out_path}')

Input: Good evening everyone, thanks for coming. John Deighton had no choice but to pursue other lines of work. I really love pizza here.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟦 TASK 1 · Claim Detection
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  -    p_claim=0.002  Good evening everyone, thanks for coming.
  -    p_claim=0.033  John Deighton had no choice but to pursue other lines of work.
  -    p_claim=0.338  I really love pizza here.

No factual claims found, nothing to verify.

Saved to /content/drive/MyDrive/NLP/Praktikum/Final Project/outputs/pipeline/last_run.json


# 📊 RESULTS SUMMARY
Reads the metrics files in Drive (only the ones that have been run).

In [26]:
stage('🟦 TASK 1 · Claim Detection (ClaimBuster test set, tuned threshold)')
for path in sorted(MODELS_DIR.rglob('metrics.json')):
    m = json.loads(path.read_text())
    if 'test@tuned' in m:
        t = m['test@tuned']
        print(f"  {m['run']:<40} thr={m['tuned_threshold']:.2f}  macroF1={t['macro_f1']:.4f}  "
              f"P={t['precision']:.4f}  R={t['recall']:.4f}  AUC={t['roc_auc']:.4f}")

stage('🟨 TASK 2 · Evidence Retrieval')
path = OUTPUTS_DIR / 'task2' / 'metrics.json'
if path.exists():
    print_retrieval_metrics(json.loads(path.read_text()))
else:
    print('  not run yet (section 2.8)')

stage('🟥 TASK 3 · Verification')
for tag in ['retrieved', 'oracle']:
    path = OUTPUTS_DIR / 'task3' / f'metrics_{tag}.json'
    if not path.exists():
        print(f'  {tag:<10} not run yet (section 3.3)')
        continue
    m = json.loads(path.read_text())
    for name, key in [('fixed thr', 'fixed_threshold'), (f"tuned thr={m['tuned_threshold']:.2f}", 'tuned_test_half')]:
        r = m[key]
        print(f"  {tag:<10} {name:<15} acc={r['accuracy']:.4f}  macroF1={r['macro_f1']:.4f}  FEVER={r['fever_score']:.4f}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟦 TASK 1 · Claim Detection (ClaimBuster test set, tuned threshold)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🟨 TASK 2 · Evidence Retrieval
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

=== Document retrieval (BM25) ===
  doc_recall@1           0.6460
  doc_recall@5           0.9000
  doc_recall@10          0.9220
  candidate_recall       0.9000  (upper bound for the reranker)

=== Sentence retrieval ===
  metric                 before rerank    after rerank
  evidence_recall@5             0.8160          0.8840
  hit@1                         0.5680          0.7960
  mrr                           0.7057          0.8631

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━